**IMPORTS NECESARIOS Y CONFIGURACIÓN GLOBAL**

Para llevar a cabo la detección y lectura de matrículas, inicialmente se importan las librerías necesarias. En esta versión el OCR clásico (PaddleOCR) se sustituye por un **modelo de visión-lenguaje (VLM)**, `Qwen2.5-VL-3B-Instruct`, cargado con `transformers`.

In [20]:
import os
import re
import csv
import json
import math
from pathlib import Path
from collections import defaultdict, deque
from time import perf_counter

import cv2
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor
from ultralytics import YOLO


**INICIALIZACIÓN GLOBAL DE MODELOS**

Definimos una única instancia global para el detector de matrículas entrenado (`best-yolo.pt`) y para el modelo VLM que lee el texto de la placa. De esta forma evitamos recargar modelos en memoria o duplicar inicializaciones entre la adquisición de métricas y el procesado de vídeo.

In [22]:
# Ruta al detector entrenado de placas
WEIGHTS_PLACAS = "weights/best-yolo.pt"

# Inicializacion del detector de matriculas compartido
detector_placas = YOLO(WEIGHTS_PLACAS)


**CARGA DEL MODELO VLM**

Se mantiene la misma estructura que el motor OCR original (`get_ocr` y `ocr_global`), pero ahora devuelve el par `(modelo, processor)` del VLM, cargado una sola vez.

In [23]:
# Instanciacion y gestion del modelo VLM global
VLM_ID = "PaddlePaddle/PaddleOCR-VL"

ocr_global = None
_ocr_disponible = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def get_ocr():
    """Devuelve (modelo, processor) del VLM global, inicializandolo solo la primera vez."""
    global ocr_global, _ocr_disponible

    if not _ocr_disponible:
        return None

    if ocr_global is None:
        try:
            processor = AutoProcessor.from_pretrained(
                VLM_ID,
                trust_remote_code=True
            )

            modelo = AutoModelForCausalLM.from_pretrained(
                VLM_ID,
                trust_remote_code=True,
                torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32
            ).to(DEVICE).eval()

            ocr_global = (modelo, processor)

        except Exception as e:
            print(f"ERROR cargando PaddleOCR-VL: {e}")
            _ocr_disponible = False
            return None

    return ocr_global


# Inicializamos el VLM globalmente
ocr_global = get_ocr()

**UTILIDADES COMUNES: GEOMETRÍA, CADENAS Y MANIPULACIÓN DE CAJAS**

A continuación se definen las funciones auxiliares compartidas necesarias para el tratamiento de texto, cálculo de solapamiento de cajas (IoU), cálculo de centros y distancias, y recorte de regiones de interés.

**UTILIDADES DE TEXTO**

Normalización de texto y distancia de Levenshtein entre cadenas.

In [24]:
# Normaliza texto a mayusculas alfanumericas
def norm(s: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", s.upper())


# Distancia de Levenshtein entre dos cadenas de caracteres
def lev(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


**UTILIDADES DE GEOMETRÍA**

Solapamiento de cajas (IoU), centroides, diagonales y recorte de regiones de interés.

In [25]:
# Devuelve el calculo del solapamiento (IoU) entre dos regiones pasadas
def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


# Centroide de una caja [x1, y1, x2, y2]
def centro(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


# Diagonal de una caja para calculo de tamano relativo
def diag(b):
    return math.hypot(b[2] - b[0], b[3] - b[1])


# Recorta una region con margen porcentual o en pixeles ajustada a los limites de la imagen
def recortar(img, box, pad=0.05, margen_px=None):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    if margen_px is not None:
        return img[max(0, y1 - margen_px):min(h, y2 + margen_px),
                   max(0, x1 - margen_px):min(w, x2 + margen_px)]
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw))
    x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh))
    y2 = min(h, int(y2 + pad * bh))
    return img[y1:y2, x1:x2]


**FUNCIÓN ÚNICA DE DETECCIÓN Y LECTURA DE MATRÍCULAS (`leer_placa`)**

Implementa la lectura de la matrícula a partir del recorte de imagen. Se mantiene el mismo preprocesamiento que en la versión con PaddleOCR (escala mínima con interpolación bicúbica, filtrado Gaussiano y padding réplica) y la lectura se delega en el VLM mediante un *prompt* fijo que pide solo los caracteres de la placa.

Como un VLM no devuelve una confianza por lectura, se estima como la **media de la probabilidad de los tokens generados**, de modo que `leer_placa` conserva la misma firma `(texto, confianza)` y el resto del código no cambia.

In [26]:

# Prompt de reconocimiento de texto
PROMPT_PLACA = "OCR:"


def leer_placa(crop, ocr_inst=None):
    """Devuelve (texto normalizado, confianza media) usando PaddleOCR-VL."""

    if crop is None or crop.size == 0:
        return "", 0.0

    reader = ocr_inst if ocr_inst is not None else get_ocr()

    if reader is None:
        return "", 0.0

    modelo, processor = reader

    # Escala minima garantizada tanto en alto como en ancho
    h_c, w_c = crop.shape[:2]
    esc = max(max(1.0, 64.0 / h_c), max(1.0, 200.0 / w_c))

    g = cv2.resize(
        crop, None, fx=esc, fy=esc,
        interpolation=cv2.INTER_CUBIC
    )

    g = cv2.GaussianBlur(g, (3, 3), 0)
    g = cv2.copyMakeBorder(
        g, 10, 10, 10, 10, cv2.BORDER_REPLICATE
    )

    # Convertimos el recorte a PIL
    pil = Image.fromarray(cv2.cvtColor(g, cv2.COLOR_BGR2RGB))

    # Preparacion de la entrada para PaddleOCR-VL
    mensajes = [{
        "role": "user",
        "content": [
            {"type": "image", "image": pil},
            {"type": "text", "text": PROMPT_PLACA}
        ]
    }]

    inputs = processor.apply_chat_template(
        mensajes,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(DEVICE)

    # Inferencia
    with torch.inference_mode():
        salida = modelo.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True
        )

    # Extraemos solamente los tokens generados
    gen = salida.sequences[:, inputs["input_ids"].shape[1]:]

    respuesta = processor.batch_decode(
        gen,
        skip_special_tokens=True
    )[0].strip()

    texto = norm(respuesta)

    if not texto or texto == "NONE":
        return "", 0.0

    # Confianza media de los tokens generados
    try:
        logp = modelo.compute_transition_scores(
            salida.sequences,
            salida.scores,
            normalize_logits=True
        )[0]

        especiales = set(processor.tokenizer.all_special_ids)

        probs = [
            float(torch.exp(lp))
            for lp, t in zip(logp, gen[0].tolist())
            if t not in especiales
        ]

        confianza = float(np.mean(probs)) if probs else 0.0

    except Exception:
        confianza = 1.0

    return texto, confianza


**TAREA 1: ADQUISICIÓN Y EVALUACIÓN DE MÉTRICAS DE LECTURA CON VLM**

En esta fase se evalúa el rendimiento cuantitativo del sistema sobre el conjunto de anotaciones reales (Ground Truth):
- **Exact Match**: Porcentaje de matrículas donde la cadena leída coincide al 100% con el texto real.
- **Acierto por carácter**: Precisión media basada en la distancia normalizada de Levenshtein: $1 - \min(1, \text{lev} / \text{longitud})$.
- **Tiempos de ejecución (`perf_counter`)**: Se mide el tiempo medio empleado por detección tanto en localización (YOLO) como en reconocimiento (VLM), así como el coste temporal total del pipeline por detección.
- **Análisis de falsos positivos**: Se diferencian las detecciones sin coincidencia con ninguna caja real (falsos positivos geométricos) de aquellas que coinciden con una placa anotada pero sin texto legible (`None`/vacía).

**CONFIGURACIÓN DEL DATASET DE EVALUACIÓN**

Rutas, umbrales y carpeta de salida donde se guardan el CSV de resultados y los recortes con error.

In [27]:
# Configuracion de rutas y parametros del dataset
BASE = Path("plates/raw-dataset")
IMAGENES = BASE / "images"
ANOTACIONES = BASE / "annotations"
SPLIT = "train"
MAX_IMAGENES = 200     # None para evaluar el dataset completo
GT_KEYS = ("lpn", "text", "plate", "label", "number", "plate_number")
CONF_METRICAS = 0.25
IOU_MATCH = 0.5
PAD = 0.05
IMGSZ_METRICAS = 960
EXTS = {".jpg", ".jpeg", ".png"}

SALIDA = Path("vlm_eval")
SALIDA.mkdir(exist_ok=True)
(SALIDA / "errores").mkdir(exist_ok=True)


**FUNCIONES AUXILIARES DE EVALUACIÓN**

Carga del Ground Truth desde los JSON (`cargar_gt`) y resumen de exact match y acierto por carácter (`resumen_metricas`).

In [28]:
# Carga las cajas y textos anotados desde los archivos JSON
def cargar_gt(img_path, w, h):
    ruta = ANOTACIONES / f"{img_path.stem}.json"
    if not ruta.exists():
        return []
    gts = []
    for obj in json.loads(ruta.read_text()):
        xc, yc, bw, bh = obj["rectangle"]
        box = ((xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h)
        txt = next((norm(str(obj[k])) for k in GT_KEYS if obj.get(k)), None)
        gts.append((box, txt))
    return gts


# Muestra el resumen de exact match y acierto medio por caracter
def resumen_metricas(nombre, rows):
    rows = [r for r in rows if r["gt"]]
    if not rows:
        return
    exact = np.mean([r["gt"] == r["ocr"] for r in rows])
    acc_car = np.mean([1 - min(1, lev(r["gt"], r["ocr"]) / max(len(r["gt"]), 1)) for r in rows])
    print(f"{nombre:<34} n={len(rows):<5} exact match={exact:.3f} | acierto por carácter={acc_car:.3f}")


A continuación se implementa la función `evaluar_dataset`. Emplea `perf_counter` para registrar con alta precisión los tiempos medios de lectura por detección (tanto a partir de las detecciones de YOLO como a partir de los recortes anotados del Ground Truth), clasifica los posibles falsos positivos geométricos frente a placas anotadas con texto `None`, y genera tanto el archivo `ocr_resultados.csv` como el guardado de recortes erróneos en disco.

In [29]:
def evaluar_dataset(lista_imagenes, detector=detector_placas):
    """Evalua OCR en detecciones YOLO y en recortes anotados (techo OCR), midiendo tiempos por deteccion con perf_counter."""
    filas = []
    n_gt = n_gt_texto = n_detectadas = 0
    tiempos_ocr_det = []
    tiempos_ocr_gt = []
    falsos_positivos_geom = 0
    detecciones_en_gt_sin_texto = 0

    for img_path in lista_imagenes:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        gts = cargar_gt(img_path, w, h)
        n_gt += len(gts)
        n_gt_texto += sum(t is not None for _, t in gts)

        # Inferencia de deteccion de placas
        r = detector.predict(img, imgsz=IMGSZ_METRICAS, conf=CONF_METRICAS, verbose=False)[0]

        preds = [(b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]
        n_detectadas += len(preds)

        # OCR a partir de las detecciones de YOLO
        for box, det_conf in preds:
            t_ini_det = perf_counter()
            rec = recortar(img, box, pad=PAD)
            texto, ocr_conf = leer_placa(rec)
            tiempos_ocr_det.append(perf_counter() - t_ini_det)

            mejor_iou, gt_txt = 0.0, None
            for gbox, gtxt in gts:
                v = iou(box, gbox)
                if v > mejor_iou:
                    mejor_iou, gt_txt = v, gtxt

            # Analisis de emparejamiento con el Ground Truth
            if mejor_iou < IOU_MATCH:
                falsos_positivos_geom += 1
                continue
            elif gt_txt is None or gt_txt == "NONE":
                detecciones_en_gt_sin_texto += 1
                continue

            filas.append(dict(imagen=img_path.name, origen="deteccion", gt=gt_txt, ocr=texto,
                              det_conf=round(det_conf, 3), ocr_conf=round(ocr_conf, 3),
                              iou=round(mejor_iou, 3)))
            if gt_txt != texto:
                cv2.imwrite(str(SALIDA / "errores" / f"{len(filas):05d}_gt-{gt_txt}_ocr-{texto}.jpg"), rec)

        # 2) OCR a partir del recorte de la anotacion = techo del OCR
        for gbox, gtxt in gts:
            if gtxt is None or gtxt == "NONE":
                continue
            t_ini_gt = perf_counter()
            rec_gt = recortar(img, gbox, pad=PAD)
            texto, ocr_conf = leer_placa(rec_gt)
            tiempos_ocr_gt.append(perf_counter() - t_ini_gt)

            filas.append(dict(imagen=img_path.name, origen="gt_recorte", gt=gtxt, ocr=texto,
                              det_conf=None, ocr_conf=round(ocr_conf, 3), iou=1.0))

    # Guardar resultados en CSV
    csv_path = SALIDA / "ocr_resultados.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(filas[0].keys()) if filas else ["vacio"])
        wr.writeheader()
        wr.writerows(filas)

    # Metricas de tiempo medio de ejecucion por deteccion (extra)
    tiempo_medio_ocr_det = (np.mean(tiempos_ocr_det) * 1000) if tiempos_ocr_det else 0.0
    tiempo_medio_ocr_gt = (np.mean(tiempos_ocr_gt) * 1000) if tiempos_ocr_gt else 0.0

    print(f"\nImágenes evaluadas: {len(lista_imagenes)} | placas anotadas: {n_gt} (con texto: {n_gt_texto}) | detecciones YOLO: {n_detectadas}")
    print(f"--- TIEMPOS MEDIOS DE LECTURA POR DETECCIÓN (perf_counter) ---")
    print(f"OCR a partir de detección YOLO:     {tiempo_medio_ocr_det:6.2f} ms / detección")
    print(f"OCR a partir de recorte GT (techo): {tiempo_medio_ocr_gt:6.2f} ms / recorte anotado")

    if n_gt_texto == 0:
        print(f"\nLas anotaciones no contienen texto de matrícula (claves probadas: {GT_KEYS}).")
    else:
        print(f"\n--- RESUMEN DE EXACTITUD OCR ---")
        resumen_metricas("OCR sobre detecciones de YOLO", [r for r in filas if r["origen"] == "deteccion"])
        resumen_metricas("OCR sobre recorte anotado (techo)", [r for r in filas if r["origen"] == "gt_recorte"])
        emp = sum(1 for r in filas if r["origen"] == "deteccion" and r["gt"])
        print(f"Placas evaluadas encontradas por YOLO: {emp}/{n_gt_texto}")
        print(f"Falsos positivos geométricos (IoU < {IOU_MATCH}): {falsos_positivos_geom}")
        print(f"Detecciones en placas anotadas sin texto (None): {detecciones_en_gt_sin_texto}")

    print(f"\nCSV: {csv_path}\nRecortes con error: {SALIDA / 'errores'}")
    return filas


**EJECUCIÓN DE LA EVALUACIÓN**

Se evalúa un subconjunto de imágenes del dataset (`MAX_IMAGENES`). Con el VLM esta celda es notablemente más lenta que con PaddleOCR.

In [30]:
# Ejecucion de la evaluacion de metricas
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    evaluar_dataset(lista)
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset.")


Directorio plates\raw-dataset\images no encontrado. Comprueba la ruta del dataset.


**TAREA 1 (EXTENSIÓN): EVALUACIÓN DEL VLM CON CORRECCIÓN POR POSICIÓN**

Se añade una segunda evaluación que reutiliza el detector, el VLM y las utilidades ya definidas (`norm`, `lev`, `iou`, `recortar`, `cargar_gt`, `leer_placa`) sin modificarlas. Respecto a `evaluar_dataset` incorpora:
- **Lectura cruda vs. lectura corregida**: se mide el VLM tal cual y tras aplicar una corrección por posición (`corregir`).
- **Tiempo por lectura en milisegundos**: media y mediana, con una lectura de calentamiento previa, ya que la primera inferencia del VLM es notablemente más lenta.
- **CSV con separador `;`** (`ocr_resultados_vlm.csv`), igual que el CSV del vídeo.

**CORRECCIÓN POR POSICIÓN Y LECTURA CRONOMETRADA**

`corregir` aprovecha que en las matrículas españolas actuales (4 dígitos + 3 consonantes) cada posición admite un solo tipo de carácter: en las posiciones numéricas se sustituyen letras confundibles (`O→0`, `S→5`, `B→8`...) y en las posiciones de letras se sustituyen dígitos confundibles (`8→B`, `5→S`...). Solo se aplica a lecturas de 7 caracteres y únicamente se acepta el resultado si cumple el formato; en caso contrario se devuelve la lectura sin tocar. `leer_cronometrado` envuelve a `leer_placa` midiendo su tiempo con `perf_counter`.

```
# Confusiones tipicas del VLM segun el tipo de posicion
CONS_ES = "BCDFGHJKLMNPRSTVWXYZ"
LETRA_A_DIGITO = {"O": "0", "Q": "0", "D": "0", "I": "1", "L": "1", "Z": "2",
                  "S": "5", "G": "6", "T": "7", "B": "8"}
DIGITO_A_LETRA = {"0": "D", "1": "L", "2": "Z", "5": "S", "6": "G", "7": "T", "8": "B"}
```

In [31]:
# Confusiones tipicas del VLM segun el tipo de posicion
CONS_ES = "BCDFGHJKLMNPRSTVWXYZ"
LETRA_A_DIGITO = {"O": "0", "Q": "0", "D": "0", "I": "1", "L": "1", "Z": "2",
                  "S": "5", "G": "6", "T": "7", "B": "8"}
DIGITO_A_LETRA = {"0": "D", "1": "L", "2": "Z", "5": "S", "6": "G", "7": "T", "8": "B"}


# Corrige una lectura de 7 caracteres al formato espanol actual (DDDDLLL) segun la posicion
def corregir(texto):
    if len(texto) != 7:
        return texto
    corr = "".join(LETRA_A_DIGITO.get(ch, ch) if k < 4 else DIGITO_A_LETRA.get(ch, ch)
                   for k, ch in enumerate(texto))
    return corr if re.fullmatch(rf"\d{{4}}[{CONS_ES}]{{3}}", corr) else texto


# Devuelve (lectura cruda, lectura corregida por posicion, milisegundos)
def leer_cronometrado(crop):
    if crop is None or crop.size == 0:
        return "", "", 0.0
    t0 = perf_counter()
    cruda, _ = leer_placa(crop)
    ms = (perf_counter() - t0) * 1000
    return cruda, corregir(cruda), ms

**FUNCIONES DE EVALUACIÓN CON LECTURA CRUDA Y CORREGIDA**

`resumen_metricas_vlm` muestra exact match, acierto por carácter ($1 - \min(1, \text{lev} / \text{longitud real})$) y tiempo medio y mediano para la columna indicada (`ocr_crudo` u `ocr_corregido`). `evaluar_dataset_vlm` evalúa tanto sobre las detecciones de YOLO (resultado de extremo a extremo) como sobre el recorte exacto de la anotación (techo del OCR, que aísla el error del detector) y guarda el CSV con separador `;`.

```
def resumen_metricas_vlm(titulo, rows, col):
    ...

def evaluar_dataset_vlm(lista_imagenes, detector=detector_placas):
    ...
```

In [ ]:
# Muestra exact match, acierto por caracter y tiempos para la columna indicada
def resumen_metricas_vlm(titulo, rows, col):
    if not rows:
        return
    exacto = np.mean([r["gt"] == r[col] for r in rows])
    por_car = np.mean([1 - min(1, lev(r["gt"], r[col]) / max(len(r["gt"]), 1)) for r in rows])
    tiempos = [r["ms"] for r in rows]
    print(f"{titulo:<40} n={len(rows):<5} exacto={exacto:.3f} | por carácter={por_car:.3f} "
          f"| {np.mean(tiempos):.0f} ms de media ({np.median(tiempos):.0f} de mediana)")


def evaluar_dataset_vlm(lista_imagenes, detector=detector_placas):
    """Evalua el VLM (crudo y corregido) sobre detecciones YOLO y sobre recortes anotados (techo)."""
    leer_cronometrado(np.zeros((30, 100, 3), np.uint8))   # calentamiento: la 1a lectura es mas lenta

    filas = []
    n_gt = n_gt_texto = n_detectadas = 0

    for img_path in lista_imagenes:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        gts = cargar_gt(img_path, w, h)
        n_gt += len(gts)
        n_gt_texto += sum(t is not None for _, t in gts)

        # Inferencia de deteccion de placas
        r = detector.predict(img, imgsz=IMGSZ_METRICAS, conf=CONF_METRICAS, verbose=False)[0]
        preds = [(b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]
        n_detectadas += len(preds)

        # 1) OCR sobre lo que detecta YOLO (resultado de extremo a extremo)
        for box, det_conf in preds:
            rec = recortar(img, box, pad=PAD)
            cruda, corregida, ms = leer_cronometrado(rec)

            mejor_iou, gt_txt = 0.0, None
            for gbox, gtxt in gts:
                v = iou(box, gbox)
                if v > mejor_iou:
                    mejor_iou, gt_txt = v, gtxt
            if mejor_iou < IOU_MATCH or gt_txt is None or gt_txt == "NONE":
                continue

            filas.append(dict(imagen=img_path.name, origen="deteccion", gt=gt_txt, ocr_crudo=cruda,
                              ocr_corregido=corregida, ms=round(ms, 1), det_conf=round(det_conf, 3),
                              iou=round(mejor_iou, 3)))
            if gt_txt != cruda:
                cv2.imwrite(str(SALIDA / "errores-yolo" / f"{len(filas):05d}_gt-{gt_txt}_ocr-{cruda}.jpg"), rec)

        # 2) OCR sobre el recorte exacto de la anotacion (techo del OCR, aisla el error de YOLO)
        for gbox, gtxt in gts:
            if gtxt is None or gtxt == "NONE":
                continue
            cruda, corregida, ms = leer_cronometrado(recortar(img, gbox, pad=PAD))
            filas.append(dict(imagen=img_path.name, origen="gt_recorte", gt=gtxt, ocr_crudo=cruda,
                              ocr_corregido=corregida, ms=round(ms, 1), det_conf=None, iou=1.0))

    # Guardar resultados en CSV (separador ;)
    csv_vlm = SALIDA / "yolo_resultados_vlm.csv"
    with open(csv_vlm, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(filas[0].keys()) if filas else ["vacio"], delimiter=";")
        wr.writeheader()
        wr.writerows(filas)

    print(f"\nImágenes: {len(lista_imagenes)} | placas anotadas: {n_gt} | detecciones YOLO: {n_detectadas}")
    if n_gt_texto == 0:
        print(f"Las anotaciones no traen texto (claves probadas: {GT_KEYS}).")
    else:
        det = [r for r in filas if r["origen"] == "deteccion"]
        techo = [r for r in filas if r["origen"] == "gt_recorte"]
        resumen_metricas_vlm("VLM sobre detecciones de YOLO", det, "ocr_crudo")
        resumen_metricas_vlm("VLM sobre recorte anotado (techo)", techo, "ocr_crudo")
        resumen_metricas_vlm("VLM + corrección, detecciones", det, "ocr_corregido")
        resumen_metricas_vlm("VLM + corrección, recorte anotado", techo, "ocr_corregido")
        print(f"Placas anotadas encontradas por YOLO: {len(det)}/{n_gt_texto}")
    print(f"\nCSV: {csv_vlm}\nRecortes con error: {SALIDA / 'errores'}")
    return filas

**EJECUCIÓN DE LA EVALUACIÓN CON CORRECCIÓN**

Usa el mismo subconjunto de imágenes (`MAX_IMAGENES`) y la misma configuración de rutas que la evaluación anterior. Si el directorio del dataset no se encuentra, se muestra un aviso: hay que revisar `BASE` en la celda de configuración del dataset.

In [33]:
# Ejecucion de la evaluacion con lectura cruda y corregida
if IMAGENES.exists():
    lista = sorted(p for p in IMAGENES.iterdir() if p.suffix.lower() in EXTS)
    if MAX_IMAGENES:
        lista = lista[:MAX_IMAGENES]
    filas_vlm = evaluar_dataset_vlm(lista)
else:
    print(f"Directorio {IMAGENES} no encontrado. Comprueba la ruta del dataset (variable BASE).")

Directorio plates\raw-dataset\images no encontrado. Comprueba la ruta del dataset (variable BASE).


**TAREA 2: DETECCIÓN, TRACKING Y LECTURA DE MATRÍCULAS EN VÍDEO (VLM)**

En este módulo se procesa una secuencia de vídeo realizando:
- **Seguimiento con ByteTrack**: Detección y seguimiento continuo de personas y vehículos (`persona`, `coche`, `moto`, `guagua`, `camion`).
- **Filtrado y validación sintáctica**: Verificación de las matrículas leídas mediante expresiones regulares de formatos de placas europeos.
- **Fusión de tracklets**: Recuperación de identificadores ante oclusiones temporales, tanto para vehículos en movimiento como para vehículos estacionados.
- **Renderizado y exportación**: Generación del vídeo anotado con los identificadores y recuadros de matrícula, junto con el archivo `resultado.csv` detallando las detecciones por fotograma.

**CONFIGURACIÓN DE RUTAS, CLASES Y PALETA**

Rutas del vídeo de entrada y de salida (`resultado.mp4` y `resultado.csv` se guardan en la carpeta `YOLO_VLM_result`), mapeo de clases COCO y colores de dibujo.

In [ ]:
# Configuracion de rutas de video
filename = "C0142.MP4"
# Carpeta de resultados: se crea en el directorio de trabajo del cuaderno
carpeta = os.path.abspath("YOLO_VLM_result")
os.makedirs(carpeta, exist_ok=True)
output_path = os.path.join(carpeta, "resultado.mp4")
csv_path = os.path.join(carpeta, "resultado.csv")

# Mapeo de clases y paleta visual
nombre = {0: "persona", 2: "coche", 3: "moto", 5: "guagua", 7: "camion"}
colores = {0: (0, 255, 0), 2: (0, 0, 255), 3: (255, 0, 255),
           5: (0, 255, 255), 7: (255, 255, 0)}

COLOR_PLACA = (0, 165, 255)
ESCALA_TEXTO_PLACA = 0.35
CONF_PLACA = 0.4
MIN_FRAMES_PLACA = 5


**PARÁMETROS DE LECTURA Y PATRONES DE MATRÍCULA**

Control de cuándo se lanza la lectura sobre cada vehículo y expresiones regulares de formatos de placas europeos.

In [35]:
# Parametros del OCR sobre video
OCR_CADA = 10
MIN_ANCHO_OCR = 40
MIN_CHARS_PLACA = 4
OCR_SCORE_OK = 0.5
OCR_MAX_INTENTOS = 3

# Patrones sintacticos de matriculas europeas
CONS = "BCDFGHJKLMNPRSTVWXYZ"
PATRONES_MATRICULA = [re.compile(p) for p in (
    rf"\d{{4}}[{CONS}]{{3}}",              # Espana actual
    r"[A-Z]{1,2}\d{4}[A-Z]{1,2}",          # Espana antigua
    r"[A-Z]{2}\d{3}[A-Z]{2}",              # Francia / Italia
    r"[A-Z]{2}\d{2}[A-Z]{3}",              # Reino Unido
    r"[A-Z]{1,3}[A-Z]{1,2}\d{1,4}[EH]?",   # Alemania
    r"\d{2}[A-Z]{2}\d{2}",                 # Portugal
    r"[A-Z]{2}\d{2}[A-Z]{2}",              # Portugal
    r"\d{2}\d{2}[A-Z]{2}",                 # Portugal
    r"[1-9][A-Z]{3}\d{3}",                 # Belgica
    r"\d{2}[A-Z]{3}\d",                    # Paises Bajos
    r"\d[A-Z]{3}\d{2}",
    r"[A-Z]{2}\d{3}[A-Z]",
    r"\d{3}[A-Z]{2}\d",
    r"[A-Z]\d{3}[A-Z]{2}",
    r"[A-Z]{3}\d{2}[A-Z]",
    r"[A-Z]{3}\d{3}",                      # Polonia / otros
    r"[A-Z]{2}\d{5}",
)]


**PARÁMETROS DE TRACKING Y FUSIÓN DE TRACKLETS**

Parámetros de fusión de identidades y configuración del archivo de ByteTrack.

In [36]:
# Parametros de tracking y fusion de tracklets
FAMILIA = {0: 'p', 3: 'm', 2: 'v', 5: 'v', 7: 'v'}
PARAM_FUSION = {'p': (5, 0.5, 0.01), 'v': (5, 0.5, 0.01), 'm': (10, 0.8, 0.02)}
GAP_ESTATICO_SEG = 120
MIN_FRAMES_ESTATICO = 15
IOU_ESTATICO = 0.5
MIN_FRAMES = 10
MAYORIA_PESADOS = 0.65
PESO_BORDE = 0.3
MARGEN_BORDE = 5

# Configuracion del archivo de ByteTrack
os.makedirs("./YOLO", exist_ok=True)
with open("./YOLO/mi_bytetrack.yaml", "w") as f:
    f.write("tracker_type: bytetrack\ntrack_high_thresh: 0.25\ntrack_low_thresh: 0.1\n"
            "new_track_thresh: 0.25\ntrack_buffer: 140\nmatch_thresh: 0.8\nfuse_score: True\n")


A continuación se implementan las funciones encargadas de:
- Validar el formato de la matrícula según los patrones definidos (`matricula_valida`).
- Extrapolar la posición temporal del objeto ante pérdida de fotogramas (`prediccion_posicion`).
- Distinguir vehículos aparcados de objetos en movimiento (`es_estatico`).
- Realizar la fusión de tracklets fragmentados por oclusión (`fusionar_tracklets`).
- Resolver la clase final del objeto por votación mayoritaria (`decidir_clase`).

In [37]:
# Valida si el texto coincide con alguna sintaxis europea conocida
def matricula_valida(texto):
    return any(p.fullmatch(texto) for p in PATRONES_MATRICULA)


# Estima la posicion esperada del tracklet tras 'gap' fotogramas perdidos
def prediccion_posicion(info, a, gap):
    (f0, c0), (f1, c1) = info[a]['hist'][0], info[a]['hist'][-1]
    if f1 == f0:
        return c1
    vx, vy = (c1[0] - c0[0]) / (f1 - f0), (c1[1] - c0[1]) / (f1 - f0)
    g = min(gap, 60)
    return (c1[0] + vx * g, c1[1] + vy * g)


# Comprueba si un objeto ha permanecido inmovil
def es_estatico(info, frames_visto, t):
    if frames_visto[t] < MIN_FRAMES_ESTATICO:
        return False
    mov = math.dist(centro(info[t]['caja_ini']), centro(info[t]['caja_fin']))
    return mov / max(diag(info[t]['caja_fin']), 1) < 0.15


**FUSIÓN DE TRACKLETS**

Unión de tracklets fragmentados por oclusión, por doble vía (vehículos aparcados o trayectoria).

In [38]:
# Algoritmo de union de tracklets por doble via (vehiculos aparcados o trayectoria)
def fusionar_tracklets(info, frames_visto, clase_prelim, fps):
    ids = sorted(info, key=lambda t: info[t]['ini'])
    raiz = {t: t for t in ids}
    con_sucesor = set()
    for b in ids:
        fam = FAMILIA[clase_prelim[b]]
        seg, base, por_frame = PARAM_FUSION[fam]
        b_estatico_ok = frames_visto[b] < MIN_FRAMES_ESTATICO or es_estatico(info, frames_visto, b)
        mejor, mejor_d = None, 1e9
        for a in ids:
            if a in con_sucesor or info[a]['fin'] >= info[b]['ini']:
                continue
            if FAMILIA[clase_prelim[a]] != fam:
                continue
            gap = info[b]['ini'] - info[a]['fin']
            da, db = diag(info[a]['caja_fin']), diag(info[b]['caja_ini'])

            # Via estatica: vehiculo aparcado con oclusion prolongada
            if (fam != 'p' and gap <= GAP_ESTATICO_SEG * fps and b_estatico_ok
                    and es_estatico(info, frames_visto, a) and 0.7 <= db / max(da, 1) <= 1.4):
                s = iou(info[a]['caja_fin'], info[b]['caja_ini'])
                if s >= IOU_ESTATICO:
                    d = 0.01 * (1 - s)
                    if d < mejor_d:
                        mejor, mejor_d = a, d
                    continue

            # Via dinamica: movimiento continuo y cercania a la posicion predicha
            if gap > seg * fps:
                continue
            if not 0.4 <= db / max(da, 1) <= 2.5:
                continue
            cb = centro(info[b]['caja_ini'])
            d = min(math.dist(centro(info[a]['caja_fin']), cb),
                    math.dist(prediccion_posicion(info, a, gap), cb)) / max(da, 1)
            if d <= base + por_frame * gap and d < mejor_d:
                mejor, mejor_d = a, d
        if mejor is not None:
            raiz[b] = raiz[mejor]
            con_sucesor.add(mejor)
    return raiz


**DECISIÓN DE LA CLASE FINAL**

In [39]:
# Decide la clase del objeto por mayoria ponderada
def decidir_clase(v):
    c = max(v, key=v.get)
    if c in (5, 7) and v[c] / sum(v.values()) < MAYORIA_PESADOS:
        return 2
    return c


Finalmente, se define la función `procesar_video` que integra las dos pasadas:
1. **Pasada 1**: Ejecución de inferencia sobre el vídeo, seguimiento con ByteTrack, detección de matrículas mediante `detector_placas`, lectura con el VLM mediante `leer_placa` y fusión temporal de identidades.
2. **Pasada 2**: Renderizado del vídeo final (`resultado.mp4`) superponiendo las cajas y matrículas leídas, y volcado a `resultado.csv`.

In [40]:
def procesar_video(video_path, salida_video, salida_csv, detector_placas_inst=detector_placas):
    """Procesamiento en dos pasadas: inferencia/tracking y renderizado/exportacion CSV."""
    model_yolo = YOLO('yolo11l.pt')

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    # ---------- Pasada 1: Deteccion, tracking y lectura OCR ----------
    print("Iniciando Pasada 1: tracking, detección y lectura OCR en vídeo...")
    results = model_yolo.track(source=video_path, stream=True, classes=list(nombre),
                               conf=0.5, iou=0.5, imgsz=960, persist=True, quantize=16,
                               verbose=False, tracker="./YOLO/mi_bytetrack.yaml")

    datos = []
    placas_datos = []
    votos = defaultdict(lambda: defaultdict(float))
    frames_visto = defaultdict(int)
    info = {}
    lecturas = {}
    ultimo_ocr = {}
    intentos_ocr = defaultdict(int)

    for i, r in enumerate(results):
        frame_data = []
        if r.boxes.id is not None:
            for caja, tid, c, conf in zip(r.boxes.xyxy.int().tolist(), r.boxes.id.int().tolist(),
                                          r.boxes.cls.int().tolist(), r.boxes.conf.tolist()):
                x1, y1, x2, y2 = caja
                borde = (x1 <= MARGEN_BORDE or y1 <= MARGEN_BORDE or
                         x2 >= w - MARGEN_BORDE or y2 >= h - MARGEN_BORDE)
                votos[tid][c] += conf * (PESO_BORDE if borde else 1.0)
                frames_visto[tid] += 1
                info.setdefault(tid, {'ini': i, 'caja_ini': caja, 'hist': deque(maxlen=15)})
                info[tid]['fin'], info[tid]['caja_fin'] = i, caja
                info[tid]['hist'].append((i, ((x1 + x2) / 2, (y1 + y2) / 2)))
                frame_data.append((caja, tid, c, conf))
        datos.append(frame_data)

        # Matriculas sobre este mismo fotograma usando detector_placas_inst
        placas_f = []
        if any(c != 0 for _, _, c, _ in frame_data):
            rp = detector_placas_inst.predict(r.orig_img, conf=CONF_PLACA, imgsz=960, quantize=16, verbose=False)[0]
            for pbox, pconf in zip(rp.boxes.xyxy.int().tolist(), rp.boxes.conf.tolist()):
                cx, cy = (pbox[0] + pbox[2]) / 2, (pbox[1] + pbox[3]) / 2
                cands = sorted(((b[2] - b[0]) * (b[3] - b[1]), t) for b, t, c, _ in frame_data
                               if c != 0 and b[0] <= cx <= b[2] and b[1] <= cy <= b[3])
                if not cands:
                    continue
                cands = [t for _, t in cands]
                t0 = cands[0]
                if (pbox[2] - pbox[0] >= MIN_ANCHO_OCR
                        and intentos_ocr[t0] < OCR_MAX_INTENTOS
                        and lecturas.get(t0, (0.0, ""))[0] < OCR_SCORE_OK
                        and (t0 not in ultimo_ocr or i - ultimo_ocr[t0] >= OCR_CADA)):
                    ultimo_ocr[t0] = i
                    intentos_ocr[t0] += 1
                    rec_placa = recortar(r.orig_img, pbox, margen_px=4)
                    texto, oconf = leer_placa(rec_placa)
                    if len(texto) >= MIN_CHARS_PLACA and matricula_valida(texto):
                        score = pconf * oconf
                        if t0 not in lecturas or score > lecturas[t0][0]:
                            lecturas[t0] = (score, texto)
                placas_f.append((pbox, pconf, cands))
        placas_datos.append(placas_f)

    # Fusion de tracklets y asignacion de lecturas OCR por vehiculo
    raiz = fusionar_tracklets(info, frames_visto, {t: max(v, key=v.get) for t, v in votos.items()}, fps)

    votos_m = defaultdict(lambda: defaultdict(float))
    frames_m = defaultdict(int)
    for tid, v in votos.items():
        for c, x in v.items():
            votos_m[raiz[tid]][c] += x
        frames_m[raiz[tid]] += frames_visto[tid]

    clase_final = {tid: decidir_clase(v) for tid, v in votos_m.items()}

    texto_veh = {}
    for t, (score, texto) in lecturas.items():
        v = raiz[t]
        if v not in texto_veh or score > texto_veh[v][0]:
            texto_veh[v] = (score, texto)
    texto_veh = {v: texto for v, (_, texto) in texto_veh.items()}

    print(f"IDs originales: {len(info)} -> tras fusionar: {len(set(raiz.values()))}")

    # ---------- Pasada 2: Renderizado de video y guardado a CSV ----------
    print("Iniciando Pasada 2: renderizado de vídeo y escritura en CSV...")
    ids_propios = {c: {} for c in nombre}
    placas_vehiculo = set()
    frames_placa = defaultdict(int)

    cap = cv2.VideoCapture(video_path)
    # Reduce la salida a 1920 px de ancho como maximo (codificar 4K es lo que mas tarda)
    esc_s = min(1.0, 1920 / w)
    w_s, h_s = int(w * esc_s) // 2 * 2, int(h * esc_s) // 2 * 2
    writer = cv2.VideoWriter(salida_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w_s, h_s))

    csv_file = open(salida_csv, "w", newline="", encoding="utf-8")
    csv_out = csv.writer(csv_file, delimiter=';')
    csv_out.writerow(["fotograma", "tipo_objeto", "confianza_objeto", "identificador_tracking",
                      "x1", "y1", "x2", "y2",
                      "matricula", "confianza_matricula", "mx1", "my1", "mx2", "my2",
                      "texto_matricula"])

    for i, (frame_data, placas) in enumerate(zip(datos, placas_datos)):
        ok, frame = cap.read()
        if not ok:
            break

        visibles = {}
        for caja, tid, _, conf in frame_data:
            vid = raiz[tid]
            if frames_m[vid] < MIN_FRAMES:
                continue
            c = clase_final[vid]
            ids_propios[c].setdefault(vid, len(ids_propios[c]) + 1)
            visibles[vid] = (caja, c, conf)

        placa_de = {}
        for pbox, pconf, cands in placas:
            vid = next((raiz[t] for t in cands
                        if raiz[t] in visibles and visibles[raiz[t]][1] != 0), None)
            if vid is None:
                continue
            frames_placa[vid] += 1
            if frames_placa[vid] < MIN_FRAMES_PLACA:
                continue
            placas_vehiculo.add(vid)
            if vid not in placa_de or pconf > placa_de[vid][1]:
                placa_de[vid] = (pbox, pconf)

        for vid, (caja, c, conf) in visibles.items():
            x1, y1, x2, y2 = caja
            idp = ids_propios[c][vid]
            etiqueta = f"{nombre[c]} {idp}"
            tiene_texto = vid in texto_veh
            cv2.rectangle(frame, (x1, y1), (x2, y2), colores[c], 2)
            cv2.putText(frame, etiqueta, (x1, max(y1 - 8, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, colores[c], 2)

            if vid in placa_de:
                (px1, py1, px2, py2), pconf = placa_de[vid]
                cv2.rectangle(frame, (px1, py1), (px2, py2), COLOR_PLACA, 2)
                txt = texto_veh[vid] if tiene_texto else "?"
                fuente, esc, grosor = cv2.FONT_HERSHEY_SIMPLEX, ESCALA_TEXTO_PLACA, 1
                (tw, th), base = cv2.getTextSize(txt, fuente, esc, grosor)
                bx1 = max(0, px1)
                by2 = max(th + base + 6, py1 - 2)
                by1 = by2 - th - base - 6
                cv2.rectangle(frame, (bx1, by1), (bx1 + tw + 8, by2), COLOR_PLACA, -1)
                cv2.putText(frame, txt, (bx1 + 4, by2 - base - 3),
                            fuente, esc, (0, 0, 0), grosor)
                fila_placa = [1, f"{pconf:.3f}", px1, py1, px2, py2, texto_veh.get(vid, "")]
            else:
                fila_placa = [0, "", "", "", "", "", ""]
            csv_out.writerow([i, nombre[c], f"{conf:.3f}", idp, x1, y1, x2, y2] + fila_placa)

        cv2.rectangle(frame, (0, h - 40), (w, h), (0, 0, 0), -1)
        texto = "  |  ".join(f"{nombre[c]}: {len(ids_propios[c])}" for c in nombre)
        texto += f"  |  matricula: {len(placas_vehiculo)}"
        texto += f"  |  VLM: {sum(v in texto_veh for v in placas_vehiculo)}"
        cv2.putText(frame, texto, (10, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        writer.write(cv2.resize(frame, (w_s, h_s)) if esc_s < 1.0 else frame)
        if i % 100 == 0:
            print(f"Pasada 2: fotograma {i}/{len(datos)}", end="\r")

    cap.release()
    writer.release()
    csv_file.close()

    print(f"Video guardado en {os.path.abspath(salida_video)} | CSV guardado en {os.path.abspath(salida_csv)}")
    print(f"Matrículas detectadas: {len(placas_vehiculo)} | Matrículas leídas con OCR: {len(texto_veh)}")

**EJECUCIÓN DEL PIPELINE DE VÍDEO**

In [41]:
# Ejecucion del pipeline de video
if os.path.exists(filename):
    procesar_video(filename, output_path, csv_path)
else:
    print(f"Vídeo {filename} no encontrado. Comprueba la ruta.")


Iniciando Pasada 1: tracking, detección y lectura OCR en vídeo...
IDs originales: 188 -> tras fusionar: 95
Iniciando Pasada 2: renderizado de vídeo y escritura en CSV...
Video guardado en c:\Users\enriq\Desktop\VC\Practicas_VC\P4\YOLO_VLM_result\resultado.mp4 | CSV guardado en c:\Users\enriq\Desktop\VC\Practicas_VC\P4\YOLO_VLM_result\resultado.csv
Matrículas detectadas: 39 | Matrículas leídas con OCR: 18
